# **Analyse missing values in data**

In [23]:
# Importing of necessary libraries
import sqlite3
import pandas as pd

In [24]:
# General connection to the DB with data
StockDataDatabase = sqlite3.connect("StockData.db")

# Index Data (NASDAQ and S&P 500)
NASDAQ = pd.read_sql("SELECT * FROM Indexes WHERE IndexName='NASDAQ'", con=StockDataDatabase, parse_dates=["Date"])
SP500 = pd.read_sql("SELECT * FROM Indexes WHERE IndexName='S&P 500'", con=StockDataDatabase, parse_dates=["Date"])

# Stocks' Tickers
tickers = pd.read_sql("SELECT * FROM Tickers", con=StockDataDatabase)

# Market Indicators
VIX = pd.read_sql("SELECT * FROM MarketIndicators WHERE Indicator='VIX'", con=StockDataDatabase, parse_dates=["Date"])
bond_13_Week = pd.read_sql("SELECT * FROM MarketIndicators WHERE Indicator='13-Week Treasury'", con=StockDataDatabase, parse_dates=["Date"])
bond_5_Year = pd.read_sql("SELECT * FROM MarketIndicators WHERE Indicator='5-Year Treasury'", con=StockDataDatabase, parse_dates=["Date"])

# Currency exchange rates
EUR_USD = pd.read_sql("SELECT * FROM CurrencyExchange WHERE Currencies='EUR/USD'", con=StockDataDatabase, parse_dates=["Date"])
JPY_USD = pd.read_sql("SELECT * FROM CurrencyExchange WHERE Currencies='JPY/USD'", con=StockDataDatabase, parse_dates=["Date"])
GBP_USD = pd.read_sql("SELECT * FROM CurrencyExchange WHERE Currencies='GBP/USD'", con=StockDataDatabase, parse_dates=["Date"])
USD_CHF = pd.read_sql("SELECT * FROM CurrencyExchange WHERE Currencies='USD/CHF'", con=StockDataDatabase, parse_dates=["Date"])
AUD_USD = pd.read_sql("SELECT * FROM CurrencyExchange WHERE Currencies='AUD/USD'", con=StockDataDatabase, parse_dates=["Date"])
USD_CAD = pd.read_sql("SELECT * FROM CurrencyExchange WHERE Currencies='USD/CAD'", con=StockDataDatabase, parse_dates=["Date"])
NZD_USD = pd.read_sql("SELECT * FROM CurrencyExchange WHERE Currencies='NZD/USD'", con=StockDataDatabase, parse_dates=["Date"])

First, we will extract dates which are present in <code>Stock Data</code>. Then we can ensure pair-wise identity between stocks.

In [25]:
dates_stock = {}

for ticker in tickers["Ticker"]:
    Dates = pd.read_sql(f"SELECT Date FROM StockData WHERE Ticker='{ticker}' AND Date>='2017-09-14'", con=StockDataDatabase, parse_dates=["Date"])
    dates_stock[ticker] = Dates.copy()

# Ensures, that all stocks dates included in the DB are pair-wise equal
for i in tickers["Ticker"]:
    for q in tickers["Ticker"]:
        if not dates_stock[q].equals(dates_stock[i]):
            print(f"There is a mimatch on {i} and {q}")


Moving on, we need to check pair-wise identity of dates present in <code>US Macroeconomical Data</code>.

In [26]:
dates_macroeconomy = []

for frame in [NASDAQ, SP500, VIX, bond_13_Week, bond_5_Year]:
    Dates = frame["Date"]
    dates_macroeconomy.append(Dates)

# This ensures pair-wise Date equality in US based macroeconomic data
for i in range(len(dates_macroeconomy)):
    for q in range(len(dates_macroeconomy)):
        if not dates_macroeconomy[i].equals(dates_macroeconomy[q]):
            print(f"There is a mimatch on {i} and {q}")

Next, we should check pair-wise identity of dates present in <code>Forex Data</code>.

In [27]:
dates_forex = []

for frame in [EUR_USD, JPY_USD, GBP_USD, USD_CHF, AUD_USD, USD_CAD, NZD_USD]:
    Dates = frame["Date"]

    dates_forex.append(Dates)

# This ensures pair-wise Date equality in Forex data
for i in range(len(dates_forex)):
    for q in range(len(dates_forex)):
        if not dates_forex[i].equals(dates_forex[q]):
            print(f"There is a mismatch on {i} and {q}")

There is likely to be inconsistency between <code>FOREX</code> data and <code>US Stock Data</code>. That is why it is needed to unify approaches to this data preparation. Most importantly, we have identity between dates in datasets for <code>US Stock Data</code> and <code>US Macroeconomy Data</code>. As we have ensured pair-wise identity of present dates among all groups of data, we can check cross-group only based on one random datasets.

But first, let's check integrity of <code>US Stock Data</code> and <code>US Macroeconomical Data</code>.

In [28]:
AAPL = pd.read_sql("SELECT Date FROM StockData WHERE Ticker='AAPL'", con=StockDataDatabase, parse_dates=["Date"])

dates_crossUS = []

for frame in [NASDAQ, AAPL]:
    Dates = frame["Date"]

    dates_crossUS.append(Dates)

# This ensures pair-wise Date equality between Macroeconomy in the US and US Stocks data
for i in range(len(dates_crossUS)):
    for q in range(len(dates_crossUS)):
        if not dates_crossUS[i].equals(dates_crossUS[q]):
            print(f"There is a mismatch on {i} and {q}")

That is why we need to unify approaches to data-cleaning in <code>Forex Data</code>. Missing dates in <code>Forex Data</code>, which are present in <code>US</code> data (so no filling on weekends) will be filled with the latest available value in FOREX. Now, we manually obtain desired data and will be used later in preparatory script <code>DataFramePrep.py</code>

In [29]:
dates_cross = []

for frame in [EUR_USD, AAPL]:
    Dates = frame["Date"]

    dates_cross.append(Dates)

mask = ~dates_cross[1].isin(dates_cross[0])
missing_dates = dates_cross[1][mask]
counter = mask.sum()

print("Missing dates:")
print(missing_dates.values)
print(f"Total missing: {counter}")


Missing dates:
['2017-11-16T00:00:00.000000000' '2019-05-22T00:00:00.000000000'
 '2025-04-21T00:00:00.000000000']
Total missing: 3
